# Notebook 01: ทดลอง Web Scraping เว็บไซต์ Yuedpao.com

โน้ตบุ๊กนี้ใช้สำหรับการทดสอบดึงข้อมูลจากเว็บไซต์ **Yuedpao (https://www.yuedpao.com/)** โดยมี 2 วิธีหลัก:
1. **Requests + BeautifulSoup4 (bs4)**: ดึง HTML แบบ Static โดยตรง
2. **Playwright (Async inside Thread)**: จำลอง Browser สำหรับเว็บ Dynamic / Single Page App พร้อมระบบเลื่อน Scroll เมาส์ลงไปด้านล่างสุดเพื่อโหลดข้อมูลแบบ Lazy Loading (แก้ไขปัญหา `NotImplementedError` บน Windows Jupyter Notebook)

## 1. วิธีที่ 1: ใช้ Requests + BeautifulSoup (bs4)

In [1]:
import requests
from bs4 import BeautifulSoup

# กำหนด Target URL และ Headers เพื่อจำลอง Browser ทั่วไป
url = "https://www.yuedpao.com/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "th,en-US;q=0.9,en;q=0.8"
}

# ส่ง Request ไปยังเว็บปลายทาง
response = requests.get(url, headers=headers, timeout=15)
print(f"Status Code: {response.status_code}")

# ใช้ BeautifulSoup อ่านและแปลง HTML
soup = BeautifulSoup(response.text, "html.parser")

# ตรวจสอบ Title และข้อมูลพื้นฐาน
page_title = soup.title.string.strip() if soup.title else "No Title"
print(f"Page Title: {page_title}")
print(f"HTML Length: {len(response.text):,} characters")

# ตัวอย่าง: ค้นหาลิงก์และรูปภาพที่พบในหน้าแรก
links = soup.find_all("a", href=True)
images = soup.find_all("img")
print(f"Found {len(links)} links, {len(images)} images in static HTML")

Status Code: 200
Page Title: Yuedpao ยืดเปล่า ยืดแต่ไม่ย้วย ศูนย์รวมเสื้อยืดแบรนด์ไทย ผ้านุ่มใส่สบาย
HTML Length: 13,453 characters
Found 0 links, 0 images in static HTML


## 2. วิธีที่ 2: ใช้ Playwright พร้อม Auto-Scroll ลงด้านล่างสุด (เวอร์ชันสำหรับ Windows Jupyter)

> **แก้ปัญหา NotImplementedError บน Windows:**
> เนื่องจาก Jupyter Notebook บน Windows รันด้วย `SelectorEventLoop` ซึ่งไม่รองรับ subprocesses ของ Playwright 
> เราจึงจำเป็นต้องย้ายการทำงานของ Playwright ไปทำใน Thread ใหม่แยกต่างหาก และบังคับใช้ `ProactorEventLoop` ใน Thread นั้น

In [2]:
import sys
import asyncio
import threading
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

def scrape_yuedpao_with_scroll(url: str = "https://www.yuedpao.com/"):
    html_content = None
    exception = None

    def worker():
        nonlocal html_content, exception
        try: 
            # 1. บังคับใช้ ProactorEventLoop บน Windows *ก่อนสร้าง Loop* เพื่อให้ระบบมองเห็นและรองรับ Subprocesses
            if sys.platform == 'win32':
                asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
                
            # 2. สร้าง Event Loop ใหม่เฉพาะสำหรับ Thread นี้
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            
            # ฟังก์ชันภายในสำหรับรัน Playwright แบบ async
            async def run_playwright():
                async with async_playwright() as p:
                    # เปิด Browser แบบ Headless
                    browser = await p.chromium.launch(headless=True)
                    page = await browser.new_page()
                    
                    # ตั้งค่าขนาดหน้าต่างเว็บ
                    await page.set_viewport_size({"width": 1280, "height": 800})
                    
                    print(f"Navigating to: {url} ...")
                    await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                    await page.wait_for_timeout(2000)
                    
                    # ทำการเลื่อน Scroll ลงไปด้านล่างสุดเรื่อยๆ เพื่อโหลดเนื้อหาแบบ Lazy Load
                    print("Scrolling down to the bottom of the page...")
                    last_height = await page.evaluate("document.body.scrollHeight")
                    
                    scroll_step = 600
                    max_scroll_attempts = 30
                    
                    for i in range(max_scroll_attempts):
                        # เลื่อนลง
                        await page.evaluate(f"window.scrollBy(0, {scroll_step})")
                        await page.wait_for_timeout(800)  # หน่วงเวลาให้ข้อมูลโหลด
                        
                        new_height = await page.evaluate("document.body.scrollHeight")
                        current_scroll_pos = await page.evaluate("window.pageYOffset + window.innerHeight")
                        
                        # ถ้าเลื่อนถึงจุดล่างสุดแล้ว
                        if current_scroll_pos >= new_height and new_height == last_height:
                            print(f"Reached bottom at attempt {i+1} (Scroll Height: {new_height}px)")
                            break
                        last_height = new_height
                    
                    # รอเพิ่มเติมให้มั่นใจว่าโหลดข้อมูลเสร็จสมบูรณ์
                    await page.wait_for_timeout(2000)
                    
                    # ดึง HTML ที่เรนเดอร์เสร็จแล้ว
                    content = await page.content()
                    await browser.close()
                    return content
            
            html_content = loop.run_until_complete(run_playwright())
        except Exception as e:
            exception = e
        finally:
            loop.close()

    # รันการทำงานของ Playwright ใน Thread แยก
    thread = threading.Thread(target=worker)
    thread.start()
    thread.join()
    
    if exception:
        raise exception
    return html_content

# เรียกใช้ฟังก์ชันเพื่อดึง HTML
html_content = scrape_yuedpao_with_scroll("https://www.yuedpao.com/")

# นำ HTML ที่ได้มาวิเคราะห์ต่อด้วย BeautifulSoup
soup_dynamic = BeautifulSoup(html_content, "html.parser")
print(f"\nDynamic HTML Length: {len(html_content):,} characters")
print(f"Page Title: {soup_dynamic.title.string.strip() if soup_dynamic.title else 'No Title'}")

# ตรวจสอบจำนวน elements ที่เรนเดอร์หลัง Scroll
dynamic_images = soup_dynamic.find_all("img")
dynamic_links = soup_dynamic.find_all("a")
print(f"Found {len(dynamic_links)} links, {len(dynamic_images)} images after dynamic scroll")

Navigating to: https://www.yuedpao.com/ ...
Scrolling down to the bottom of the page...
Reached bottom at attempt 11 (Scroll Height: 6992px)

Dynamic HTML Length: 414,884 characters
Page Title: Yuedpao ยืดเปล่า ยืดแต่ไม่ย้วย ศูนย์รวมเสื้อยืดแบรนด์ไทย ผ้านุ่มใส่สบาย
Found 162 links, 129 images after dynamic scroll


## 3. วิธีที่ 3: ค้นหาแถบเมนูด้านข้าง (Hamburger Menu Drawer) และตรวจหา Class เฉพาะ

ในส่วนนี้เราจะทำการดึงหน้าเว็บด้วย viewport แบบ Mobile เพื่อให้ปุ่ม Hamburger Menu แสดงขึ้นมา จากนั้นจะจำลองการกดเปิด Drawer เพื่อไปค้นหาว่ามี class ที่มีชื่อว่า `MuiTypography-root MuiTypography-body pointer-cursor css-1dwwjt3` สำหรับรายการเมนู เช่น **ULTRA FLOW (กีฬา)** อยู่ทั้งหมดกี่อัน

In [3]:
import sys
import asyncio
import threading
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

def scrape_yuedpao_mobile_drawer(url: str = "https://www.yuedpao.com/"):
    html_content = None
    exception = None

    def worker():
        nonlocal html_content, exception
        try:
            # 1. ต้องตั้งค่านโยบาย Event Loop ก่อนสร้าง Loop เสมอ เพื่อแก้ปัญหาบน Windows Jupyter
            if sys.platform == 'win32':
                asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
                
            # 2. สร้าง Event Loop ขึ้นมาใหม่
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            
            async def run():
                async with async_playwright() as p:
                    browser = await p.chromium.launch(headless=True)
                    page = await browser.new_page()
                    
                    # จำลองเปิดเป็นหน้าจอมือถือเพื่อรองรับ Hamburger Menu
                    await page.set_viewport_size({"width": 375, "height": 812})
                    
                    print(f"Navigating to: {url} on mobile viewport...")
                    await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                    await page.wait_for_timeout(3000)
                    
                    # หา SVG Hamburger Menu ที่มี Path ที่ระบุ
                    svgs = await page.query_selector_all("svg")
                    clicked = False
                    for svg in svgs:
                        # ค้นหา SVG ที่มีค่า path d ตรงกับปุ่มเปิด Menu ข้างของเว็บ
                        has_path = await svg.evaluate("""
                            (el) => {
                                const path = el.querySelector('path');
                                return path && path.getAttribute('d') === 'M13,16H3a1,1,0,0,0,0,2H13a1,1,0,0,0,0-2ZM3,8H21a1,1,0,0,0,0-2H3A1,1,0,0,0,3,8Zm18,3H3a1,1,0,0,0,0,2H21a1,1,0,0,0,0-2Z';
                            }
                        """)
                        if has_path:
                            print("Hamburger Menu button found! Clicking to open side navigation...")
                            await svg.click()
                            await page.wait_for_timeout(2500)  # รอ Drawer กางออกมาจนสุด
                            clicked = True
                            break
                    
                    if not clicked:
                        print("Warning: Hamburger Menu SVG path not found/clicked")
                    
                    content = await page.content()
                    await browser.close()
                    return content
            
            html_content = loop.run_until_complete(run())
        except Exception as e:
            exception = e
        finally:
            loop.close()

    thread = threading.Thread(target=worker)
    thread.start()
    thread.join()
    
    if exception:
        raise exception
    return html_content

In [4]:
# ดึงข้อมูล HTML จากเวอร์ชัน Mobile หลังเปิด Drawer
mobile_html = scrape_yuedpao_mobile_drawer("https://www.yuedpao.com/")
soup_mobile = BeautifulSoup(mobile_html, "html.parser")

# 1. ค้นหาทุก span ที่ใช้คลาสสิกของเมนูย่อย
# ในที่นี้คือ: class="MuiTypography-root MuiTypography-body pointer-cursor css-1dwwjt3"
target_class = ["MuiTypography-root", "MuiTypography-body", "pointer-cursor", "css-1dwwjt3"]

# ดึงเฉพาะ span ที่มีคลาสเหล่านี้ครบ
matching_spans = []
for span in soup_mobile.find_all("span"):
    classes = span.get("class", [])
    # ตรวจสอบว่าคลาสสำคัญมีครบถ้วน
    if all(cls in classes for cls in target_class):
        matching_spans.append(span)

print(f"\nTotal spans matching classes {target_class}: {len(matching_spans)}")
print("--------------------------------------------------")

for s in matching_spans:
    text = s.get_text(strip=True)
    # แสดงเมนูทั้งหมดที่ค้นพบ
    print(f"- Menu Item Found: {text}")

Navigating to: https://www.yuedpao.com/ on mobile viewport...
Hamburger Menu button found! Clicking to open side navigation...

Total spans matching classes ['MuiTypography-root', 'MuiTypography-body', 'pointer-cursor', 'css-1dwwjt3']: 34
--------------------------------------------------
- Menu Item Found: ULTRA FLOW (กีฬา)
- Menu Item Found: ULTRASOFT NON-IRON (เสื้อยืด)
- Menu Item Found: ULTIMATE COLLECTION (Work wear)
- Menu Item Found: SMOOTH SKIN
- Menu Item Found: SMOOTH FLEX JEANS (กางเกงยีนส์ยืด)
- Menu Item Found: SMOOTH STRETCH JEANS
- Menu Item Found: SOFT TECH UNWEAR
- Menu Item Found: UNWEAR
- Menu Item Found: RIB BRA
- Menu Item Found: RIB COOL MOOD
- Menu Item Found: KODNUM
- Menu Item Found: SIGNATURE
- Menu Item Found: OVERSIZED
- Menu Item Found: OVERSIZE TIMELESS
- Menu Item Found: FEATHER COMFORT
- Menu Item Found: YUEDPAO COLLECTION
- Menu Item Found: FLEECE AIR FLOW COLLECTION
- Menu Item Found: COLLAB COLLECTION
- Menu Item Found: ECOTECH
- Menu Item Found: YXZ

## 4. วิธีที่ 4: ดึงข้อมูลโครงสร้างเมนูย่อยด้านใน (Nested Menu Scraper)

ในขั้นตอนนี้ เราจะทำการแยกย่อยโค้ดออกเป็นส่วนๆ เพื่อให้ง่ายต่อการอ่านและรันศึกษาการทำงานทีละขั้น

### ขั้นตอนที่ 4.1: ดึงเฉพาะรายชื่อหมวดหมู่หลัก (Main Catalog Categories)
เราจะนำรายชื่อเมนูหลักที่แสดงบนแถบ Drawer ด้านข้าง (จากผลลัพธ์ของ BeautifulSoup ใน Cell ก่อนหน้า) มาทำการกรองเอาเฉพาะหมวดหมู่ที่เป็นหมวดสินค้าจริง

In [44]:
# กรองรายการเมนูหลักเฉพาะกลุ่มประเภทสินค้า (ตัดอีเมล, ลิงก์นโยบาย และเมนูบริการลูกค้าออก)
main_categories = []
for s in matching_spans:
    text = s.get_text(strip=True)
    if text and "cs_yuedpao" not in text and text not in ["เกี่ยวกับ", "นโยบาย", "ช่วยเหลือ", "สนับสนุน"]:
        main_categories.append(text)

print(f"จำนวนหมวดหมู่หลักที่จะไปค้นหาเมนูย่อย: {len(main_categories)} หมวดหมู่")
print("รายชื่อหมวดหมู่หลัก:", main_categories)

จำนวนหมวดหมู่หลักที่จะไปค้นหาเมนูย่อย: 34 หมวดหมู่
รายชื่อหมวดหมู่หลัก: ['ULTRA FLOW (กีฬา)', 'ULTRASOFT NON-IRON (เสื้อยืด)', 'ULTIMATE COLLECTION (Work wear)', 'SMOOTH SKIN', 'SMOOTH FLEX JEANS (กางเกงยีนส์ยืด)', 'SMOOTH STRETCH JEANS', 'SOFT TECH UNWEAR', 'UNWEAR', 'RIB BRA', 'RIB COOL MOOD', 'KODNUM', 'SIGNATURE', 'OVERSIZED', 'OVERSIZE TIMELESS', 'FEATHER COMFORT', 'YUEDPAO COLLECTION', 'FLEECE AIR FLOW COLLECTION', 'COLLAB COLLECTION', 'ECOTECH', 'YXZ COLLABORATION', 'SMOOTH STRETCH CARGO', 'POLO WAFFLE', 'RUNNING ROULETTE', 'ACCESSORIES', 'Minimal Street Collection', 'TAILOR COOL POLO INNOVATION', 'SWEATER', 'OVERSIZE BOXY', 'CHRISTMAS 2025', 'End of year sale 50%', 'End of year sale 40%', 'End of year sale 30%', 'End of year sale 60%', 'End of year sale 20%']


### ขั้นตอนที่ 4.2: สร้างฟังก์ชันย่อยสำหรับดึงเมนูย่อยรายหมวดหมู่ (Get Submenu Items)
สร้างฟังก์ชัน `get_submenu_items` เพื่อเปิด Playwright ไปจำลองการกดคลิกเข้าไปที่เมนูหลักทีละอัน เพื่ออ่านเมนูย่อยที่เรนเดอร์ใน Drawer

In [45]:
import sys
import asyncio
import threading
from playwright.async_api import async_playwright

def get_submenu_items(main_menu_name: str, main_categories_list: list, url: str = "https://www.yuedpao.com/"):
    """
    เปิด Playwright, กาง Drawer, ค้นหาเมนูหลักแล้วคลิก จากนั้นอ่านเมนูย่อยที่แสดงผลอยู่ใน Drawer ปัจจุบัน 
    โดยทำการกรองรายการเมนูหลักเดิมและข้อความทั่วไปทิ้งไป
    """
    sub_items = []
    exception = None

    def worker():
        nonlocal sub_items, exception
        try:
            # 1. ต้องตั้งค่านโยบาย Event Loop ก่อนสร้าง Loop เสมอ เพื่อแก้ปัญหาบน Windows Jupyter
            if sys.platform == 'win32':
                asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
                
            # 2. สร้าง Event Loop ขึ้นมาใหม่จากนโยบายที่อัปเดตแล้ว
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            
            async def run():
                async with async_playwright() as p:
                    browser = await p.chromium.launch(headless=True)
                    page = await browser.new_page()
                    await page.set_viewport_size({"width": 375, "height": 812})
                    
                    # ไปยังเว็บปลายทาง
                    await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                    await page.wait_for_timeout(3000)
                    
                    # คลิกเปิด Hamburger Menu Drawer
                    svgs = await page.query_selector_all("svg")
                    drawer_opened = False
                    for svg in svgs:
                        has_path = await svg.evaluate("""
                            (el) => {
                                const path = el.querySelector('path');
                                return path && path.getAttribute('d') === 'M13,16H3a1,1,0,0,0,0,2H13a1,1,0,0,0,0-2ZM3,8H21a1,1,0,0,0,0-2H3A1,1,0,0,0,3,8Zm18,3H3a1,1,0,0,0,0,2H21a1,1,0,0,0,0-2Z';
                            }
                        """)
                        if has_path:
                            await svg.click()
                            await page.wait_for_timeout(2000)
                            drawer_opened = True
                            break
                            
                    if not drawer_opened:
                        await browser.close()
                        return
                        
                    # ค้นหาข้อความเมนูหลักในบรรดา element ที่แสดงผลอยู่ ณ ตอนนั้น
                    spans = await page.query_selector_all("span.css-1dwwjt3")
                    target_item = None
                    for s in spans:
                        if await s.is_visible():
                            t = (await s.inner_text()).strip()
                            if t == main_menu_name:
                                target_item = s
                                break
                                
                    if target_item:
                        # คลิกเข้าไปในหมวดหมู่
                        await target_item.click()
                        await page.wait_for_timeout(2000)
                        
                        # ดึงเฉพาะรายการย่อยที่กางแสดงผลขึ้นมาใหม่ (คลาส css-1dwwjt3 และสามารถมองเห็นได้)
                        sub_spans = await page.query_selector_all("span.css-1dwwjt3")
                        for ss in sub_spans:
                            if await ss.is_visible():
                                sub_text = (await ss.inner_text()).strip()
                                
                                # กรองเมนูที่ไม่ใช่เมนูย่อยจริงๆ ออก เช่น:
                                # 1. หัวข้อเมนูหลักที่เป็นของตัวเอง
                                # 2. ปุ่ม 'ดูทั้งหมด'
                                # 3. รายการเมนูหลักเดิมอื่นๆ ทั้งหมด (แก้ปัญหาติดเมนูระดับนอกมา)
                                # 4. เมนูสนับสนุนทั่วไปและอีเมลติดต่อ
                                if (sub_text and 
                                    sub_text != main_menu_name and 
                                    sub_text != "ดูทั้งหมด" and 
                                    "cs_yuedpao" not in sub_text and 
                                    sub_text not in main_categories_list and 
                                    sub_text not in ["เกี่ยวกับ", "นโยบาย", "ช่วยเหลือ", "สนับสนุน"]):
                                    sub_items.append(sub_text)
                                    
                    await browser.close()
            loop.run_until_complete(run())
        except Exception as e:
            exception = e
        finally:
            loop.close()

    thread = threading.Thread(target=worker)
    thread.start()
    thread.join()
    
    if exception:
        raise exception
    return sub_items

### ขั้นตอนที่ 4.3: รันการขูดข้อมูลทุกหมวดหมู่ และนำเสนอเป็นโครงสร้าง Markdown
เพื่อป้องกันความล่าช้าในการดึงข้อมูลทั้งหมด 30+ หมวดหมู่ในขั้นตอนการทดลอง ในโค้ดตัวอย่างนี้จะรันการดึงเมนูย่อยเฉพาะ **5 หมวดหมู่แรก** 
> **Tip:** หากต้องการดึงครบถ้วนทั้งหมดในรอบจริง ให้ทำการเปลี่ยน `categories_to_run = main_categories[:5]` เป็น `categories_to_run = main_categories` ในบรรทัดแรกของโค้ด

In [46]:
from IPython.display import display, Markdown

# ดึงตัวอย่าง 5 หมวดหมู่แรก เพื่อความรวดเร็วในการรันสาธิต
categories_to_run = main_categories[:5]
print(f"เริ่มต้นประมวลผลเมนูย่อยสำหรับ: {categories_to_run}\n")

nested_results = {}
for cat in categories_to_run:
    print(f"Scraping submenus of '{cat}'...")
    try:
        # ส่งรายชื่อเมนูหลักทั้งหมด (main_categories) เข้าไปกรองแยกในฟังก์ชัน
        sub_menus = get_submenu_items(cat, main_categories)
        nested_results[cat] = sub_menus
        print(f"   -> ดึงสำเร็จ! พบเมนูย่อย {len(sub_menus)} รายการ")
    except Exception as e:
        print(f"   -> เกิดข้อผิดพลาดกับหมวดหมู่ '{cat}': {e}")
        nested_results[cat] = []

# --------------------------------------------------
# สร้าง Markdown สรุปโครงสร้างเมนูเพื่อแสดงผล
# --------------------------------------------------
markdown_lines = ["# โครงสร้างเมนูหลักและเมนูย่อยของยืดเปล่า (Nested Menus)"]
for parent, children in nested_results.items():
    markdown_lines.append(f"- **{parent}**")
    if children:
        for child in children:
            markdown_lines.append(f"  - {child}")
    else:
        markdown_lines.append("  - *(ไม่มีเมนูย่อย)*")

markdown_str = "\n".join(markdown_lines)

# แสดงผลลัพธ์แบบ Markdown
display(Markdown(markdown_str))

# ปรินต์ข้อความ Markdown ดิบให้สามารถนำไปใช้งานต่อได้
print("\n=== RAW MARKDOWN OUTPUT ===")
print(markdown_str)

เริ่มต้นประมวลผลเมนูย่อยสำหรับ: ['ULTRA FLOW (กีฬา)', 'ULTRASOFT NON-IRON (เสื้อยืด)', 'ULTIMATE COLLECTION (Work wear)', 'SMOOTH SKIN', 'SMOOTH FLEX JEANS (กางเกงยีนส์ยืด)']

Scraping submenus of 'ULTRA FLOW (กีฬา)'...
   -> ดึงสำเร็จ! พบเมนูย่อย 3 รายการ
Scraping submenus of 'ULTRASOFT NON-IRON (เสื้อยืด)'...
   -> ดึงสำเร็จ! พบเมนูย่อย 12 รายการ
Scraping submenus of 'ULTIMATE COLLECTION (Work wear)'...
   -> ดึงสำเร็จ! พบเมนูย่อย 5 รายการ
Scraping submenus of 'SMOOTH SKIN'...
   -> ดึงสำเร็จ! พบเมนูย่อย 3 รายการ
Scraping submenus of 'SMOOTH FLEX JEANS (กางเกงยีนส์ยืด)'...
   -> ดึงสำเร็จ! พบเมนูย่อย 7 รายการ


# โครงสร้างเมนูหลักและเมนูย่อยของยืดเปล่า (Nested Menus)
- **ULTRA FLOW (กีฬา)**
  - MotionSkin Active Were
  - ULTRA FLOW 2026
  - ULTRA FLOW 2025
- **ULTRASOFT NON-IRON (เสื้อยืด)**
  - Unisex Round Neck (คอกลม)2026
  - Unisex V Neck(คอวี) 2026
  - Woman Round Neck (คอกลม) 2026
  - Woman V Neck(คอวี) 2026
  - Unisex Longsleeve(แขนยาว) 2026
  - Unisex Round Neck 2025
  - Unisex V Neck 2025
  - Unisex Longsleeve 2025
  - Ultrasoft Unisex Kid
  - Woman Round Neck
  - Ultrasoft Unisex Kid 2026
  - Woman Mini T-Shirt
- **ULTIMATE COLLECTION (Work wear)**
  - Ultimate Stretch Corduroy
  - Ultimate Stretch Men (เสื้อเชิ้ตผู้ชาย)
  - Ultimate Stretch Women (เสื้อเชิ้ตผู้หญิง)
  - Men's Ultimate Pants
  - Women's Ultimate Pants
- **SMOOTH SKIN**
  - Smooth Skin2026
  - Smooth Skin LongSleeve2026
  - Smooth Skin BabyTee2026
- **SMOOTH FLEX JEANS (กางเกงยีนส์ยืด)**
  - Smooth Flex Double Pleated Jeans
  - Straight Leg Jeans
  - Smooth Flex Leg Kids Jeans
  - Smooth Flex Wide Leg Women
  - Smooth Flex Straight Leg Jeans Kids
  - Smooth Flex Flare Denim Shorts 2026
  - Smooth Flex Bermuda Short Jeans 2026


=== RAW MARKDOWN OUTPUT ===
# โครงสร้างเมนูหลักและเมนูย่อยของยืดเปล่า (Nested Menus)
- **ULTRA FLOW (กีฬา)**
  - MotionSkin Active Were
  - ULTRA FLOW 2026
  - ULTRA FLOW 2025
- **ULTRASOFT NON-IRON (เสื้อยืด)**
  - Unisex Round Neck (คอกลม)2026
  - Unisex V Neck(คอวี) 2026
  - Woman Round Neck (คอกลม) 2026
  - Woman V Neck(คอวี) 2026
  - Unisex Longsleeve(แขนยาว) 2026
  - Unisex Round Neck 2025
  - Unisex V Neck 2025
  - Unisex Longsleeve 2025
  - Ultrasoft Unisex Kid
  - Woman Round Neck
  - Ultrasoft Unisex Kid 2026
  - Woman Mini T-Shirt
- **ULTIMATE COLLECTION (Work wear)**
  - Ultimate Stretch Corduroy
  - Ultimate Stretch Men (เสื้อเชิ้ตผู้ชาย)
  - Ultimate Stretch Women (เสื้อเชิ้ตผู้หญิง)
  - Men's Ultimate Pants
  - Women's Ultimate Pants
- **SMOOTH SKIN**
  - Smooth Skin2026
  - Smooth Skin LongSleeve2026
  - Smooth Skin BabyTee2026
- **SMOOTH FLEX JEANS (กางเกงยีนส์ยืด)**
  - Smooth Flex Double Pleated Jeans
  - Straight Leg Jeans
  - Smooth Flex Leg Kids Jeans
  - Smooth 

## 5. วิธีที่ 5: การดึงข้อมูลรายการสินค้าตามหมวดหมู่ (Product List Scraper with Pagination)

ในส่วนนี้เราจะจำลองการเข้าไปหน้าหมวดหมู่สินค้าโดยระบุ URL (เช่น Unisex Round Neck คอกลม 2026) จากนั้นทำการดึงข้อมูลสินค้าทั้งหมดภายใต้การ์ดแต่ละใบ พร้อมระบบเลื่อนลงด้านล่าง (Scroll) เพื่อโหลดของแบบ Lazy Loading และกดคลิกปุ่มเปลี่ยนหน้าถัดไป (Pagination) จนครบล่าสุด

### ขั้นตอนที่ 5.1: สร้างฟังก์ชันสกัดข้อมูลสินค้ารายตัวจาก HTML (Parse Products Helper)
สร้างฟังก์ชัน `parse_products_from_html` เพื่อดึงรหัสสินค้า (Product ID), ชื่อสินค้า (Product Name), ราคาสินค้า (Price) และลิงก์รูปภาพตัวอย่างของการ์ดคลาส `vertical-product-item`

In [47]:
import re
from bs4 import BeautifulSoup

def parse_products_from_html(html_content: str):
    """
    ใช้ BeautifulSoup สกัดข้อมูลสินค้ายืดเปล่าจากการ์ดสินค้า
    """
    soup = BeautifulSoup(html_content, "html.parser")
    products = []
    
    # ค้นหาการ์ดสินค้าทั้งหมดที่มีคลาส vertical-product-item
    items = soup.find_all("div", class_=lambda x: x and "vertical-product-item" in x)
    
    for item in items:
        product_id = item.get("data-product-id", "")
        
        # 1. ดึงชื่อสินค้า (ค้นหา div ที่ควบคุมการแสดงผลและตัดบรรทัด เช่น class line-clamp-2)
        name_div = item.find("div", class_=lambda x: x and "line-clamp-2" in x)
        name = name_div.get_text(strip=True) if name_div else ""
        
        # 2. ดึงราคาสินค้า (ค้นหา p ที่เก็บราคาสินค้าหลักของตัวเลือก เช่น คลาส text-ci-primary หรือ css-6b2fbd)
        price_text = ""
        price_p = item.find("p", class_=lambda x: x and ("text-ci-primary" in x or "css-6b2fbd" in x))
        if price_p:
            price_text = price_p.get_text(strip=True)
        else:
            # Fallback ค้นหาข้อความที่มีสัญลักษณ์เงินบาท ฿
            p_fallback = item.find(lambda tag: tag.name in ["p", "div", "span"] and "฿" in tag.text)
            if p_fallback:
                price_text = p_fallback.get_text(strip=True)
                
        # แปลงราคาให้เป็นตัวเลขหลักจำนวนเต็ม (เช่น ฿ 100 -> 100)
        price = re.sub(r"[^\d]", "", price_text)
        price = int(price) if price else 0
        
        # 3. ดึงลิงก์รูปภาพของสินค้า (ค้นหา div ที่สไตล์ background-image: url(...))
        image_url = ""
        img_div = item.find("div", style=lambda x: x and "background-image" in x)
        if img_div and "style" in img_div.attrs:
            style_str = img_div["style"]
            # ใช้ regex ค้นหา URL ด้านในวงเล็บ
            match = re.search(r'url\((?:&quot;|"|\')?(.*?)(?:&quot;|"|\')?\)', style_str)
            if match:
                image_url = match.group(1)
                
        products.append({
            "product_id": product_id,
            "name": name,
            "price": price,
            "image_url": image_url
        })
        
    return products

### ขั้นตอนที่ 5.2: สร้างฟังก์ชันสกรอลหน้าและเปลี่ยนหน้าอัตโนมัติ (Playwright Crawler & Paginate)
สร้างฟังก์ชัน `scrape_product_catalog` ที่จะเลื่อนหน้าจอลงเพื่อโหลด Lazy Load ของสินค้า และตรวจสอบปุ่มเปลี่ยนหน้าถัดไป (Pagination Next Button) หากปุ่มยังไม่ถูกปิดใช้งาน (Mui-disabled) จะคลิกเพื่อทำงานหน้าถัดไปเรื่อยๆ จนจบการเก็บข้อมูล

In [48]:
import sys
import asyncio
import threading
from playwright.async_api import async_playwright

def scrape_product_catalog(url: str):
    """
    ดึงข้อมูลสินค้าทั้งหมดภายใต้ URL หมวดหมู่ เลื่อนสกรอลหน้าจอ และเปลี่ยนหน้าจนจบสุดสายนโยบาย
    """
    all_products = []
    exception = None

    def worker():
        nonlocal all_products, exception
        try:
            if sys.platform == 'win32':
                asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
                
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            
            async def run():
                async with async_playwright() as p:
                    browser = await p.chromium.launch(headless=True)
                    page = await browser.new_page()
                    await page.set_viewport_size({"width": 1280, "height": 800})
                    
                    current_url = url
                    page_num = 1
                    
                    while True:
                        print(f"Navigating to Page {page_num}: {current_url}")
                        await page.goto(current_url, wait_until="domcontentloaded", timeout=60000)
                        await page.wait_for_timeout(3000)
                        
                        # เลื่อนหน้าจอลงไปข้างล่างสุดช้าๆ เพื่อให้ Lazy Load ทำงานครบถ้วน
                        print("   Scrolling to load elements...")
                        last_height = await page.evaluate("document.body.scrollHeight")
                        scroll_step = 700
                        for _ in range(15):
                            await page.evaluate(f"window.scrollBy(0, {scroll_step})")
                            await page.wait_for_timeout(500)
                            new_height = await page.evaluate("document.body.scrollHeight")
                            current_scroll_pos = await page.evaluate("window.pageYOffset + window.innerHeight")
                            if current_scroll_pos >= new_height and new_height == last_height:
                                break
                            last_height = new_height
                            
                        await page.wait_for_timeout(1500)
                        
                        # ดึง HTML มาสกัดด้วย BeautifulSoup
                        html_content = await page.content()
                        products_in_page = parse_products_from_html(html_content)
                        print(f"   -> Page {page_num}: Found {len(products_in_page)} products")
                        all_products.extend(products_in_page)
                        
                        # ค้นหาปุ่มไปหน้าถัดไป (Pagination Next Button)
                        # อ้างอิงจากปุ่มที่มี aria-label="Go to next page"
                        next_button = await page.query_selector('a[aria-label="Go to next page"]')
                        if not next_button:
                            print("   -> Pagination Next Button not found. Reached the end.")
                            break
                            
                        # ตรวจเช็กคลาสสถานะปุ่ม (หากปุ่มถูกปิดใช้งาน Mui-disabled / aria-disabled="true")
                        is_disabled = await next_button.evaluate("""
                            (el) => el.classList.contains('Mui-disabled') || el.getAttribute('aria-disabled') === 'true'
                        """)
                        
                        if is_disabled:
                            print("   -> Next button is disabled (Reached the last page).")
                            break
                            
                        # ดึงลิงก์ของหน้าถัดไปเพื่อนำมาประกอบเข้ากับ URL หลัก
                        next_href = await next_button.get_attribute("href")
                        if not next_href:
                            print("   -> Next href not found.")
                            break
                            
                        # ประกอบ URL เพื่อใช้นำทางในรอบถัดไป
                        if next_href.startswith("/"):
                            current_url = "https://www.yuedpao.com" + next_href
                        else:
                            current_url = next_href
                            
                        page_num += 1
                        
                    await browser.close()
            loop.run_until_complete(run())
        except Exception as e:
            exception = e
        finally:
            loop.close()

    thread = threading.Thread(target=worker)
    thread.start()
    thread.join()
    
    if exception:
        raise exception
    return all_products

### ขั้นตอนที่ 5.3: สั่งรันแคตตาล็อกสแครปสินค้าตัวอย่าง และนำมาแสดงผลเป็นตาราง (Run Catalog Scraper)
ป้อน URL หมวดหมู่สินค้าคอกลม Unisex Round Neck 2026 ตัวอย่างของยืดเปล่าเพื่อดูผลลัพธ์การดึงข้อมูลออกมาแสดงบน Pandas DataFrame

In [49]:
import pandas as pd

# URL ตัวอย่างหน้า Unisex Round Neck (คอกลม) 2026 ของยืดเปล่า
target_cat_url = "https://www.yuedpao.com/UnisexRoundNeck(%E0%B8%84%E0%B8%AD%E0%B8%81%E0%B8%A5%E0%B8%A1)2026-cat.0ycq8v-92pdg6?sorter=PRODUCT_SORTER_POPULAR"

print("Starting Product Catalog Scraper...")
products_collected = scrape_product_catalog(target_cat_url)
print(f"\nScrape Finished! Total raw products collected: {len(products_collected)}")

if products_collected:
    df = pd.DataFrame(products_collected)
    # ทำความสะอาดข้อมูล: ตัด ID สินค้าซ้ำ
    df_unique = df.drop_duplicates(subset=["product_id"])
    print(f"Total unique products collected: {len(df_unique)}")
    
    # แสดงตัวอย่างข้อมูลสินค้า 10 แถวแรก
    display(df_unique.head(10))
else:
    print("No products collected.")

Starting Product Catalog Scraper...
Navigating to Page 1: https://www.yuedpao.com/UnisexRoundNeck(%E0%B8%84%E0%B8%AD%E0%B8%81%E0%B8%A5%E0%B8%A1)2026-cat.0ycq8v-92pdg6?sorter=PRODUCT_SORTER_POPULAR
   Scrolling to load elements...
   -> Page 1: Found 16 products
   -> Next button is disabled (Reached the last page).

Scrape Finished! Total raw products collected: 16
Total unique products collected: 16


,product_id,name,price,image_url
0,ur4hhrx0qyeh1n9a77qh,Ultrasoft Unisex Round Neck สี Creamy,100,https://mp-static.yuedpao.com/physical/cover/ur4hhrx0qyeh1n9a77qh/image/llnxcfiv
1,azmu24o6p9b9911c939d,Ultrasoft Unisex Round Neck สี Pink Cotton,100,https://mp-static.yuedpao.com/physical/cover/azmu24o6p9b9911c939d/image/hw36ivkx
2,kny6o61matx2wonwy6yn,Ultrasoft Unisex Round Neck สี Sunray,100,https://mp-static.yuedpao.com/physical/cover/kny6o61matx2wonwy6yn/image/zaenzrtp
3,f32vhbr8gezdb8n8jltm,Ultrasoft Unisex Round Neck สี Maroon,100,https://mp-static.yuedpao.com/physical/cover/f32vhbr8gezdb8n8jltm/image/qgfqti6h
4,sw3y1eqoky0n5tln7tjq,Ultrasoft Unisex Round Neck สี White,100,https://mp-static.yuedpao.com/physical/cover/sw3y1eqoky0n5tln7tjq/image/trq17kjn
5,wme8hz8w1vkpd9j1077p,Ultrasoft Unisex คอกลม_Light Gray,100,https://mp-static.yuedpao.com/physical/cover/wme8hz8w1vkpd9j1077p/image/hdrhdm2c
6,1kxp6bvx5fpgej35l3x5,Ultrasoft Unisex Round Neck สี Snow Blue,100,https://mp-static.yuedpao.com/physical/cover/1kxp6bvx5fpgej35l3x5/image/ah1p4hyd
7,zqv0aaftonu5wq2q1eh1,Ultrasoft Unisex คอกลม_Light Lavender,100,https://mp-static.yuedpao.com/physical/cover/zqv0aaftonu5wq2q1eh1/image/sde8z36y
8,up18y5q3oivd3g23k69c,Ultrasoft Unisex คอกลม_Dark Gray,100,https://mp-static.yuedpao.com/physical/cover/up18y5q3oivd3g23k69c/image/07mkmq23
9,qwuqdlhgjgciahcc0w3y,Ultrasoft Unisex คอกลม_Coffee Brown,100,https://mp-static.yuedpao.com/physical/cover/qwuqdlhgjgciahcc0w3y/image/pxx3txfk


## 6. วิธีที่ 6: การดึงรายละเอียดสินค้าเชิงลึกและการแสดงผลเป็นตารางข้อมูล (Product Detail Scraper & DataFrame Display)

ในส่วนนี้เราจะทำการดึงข้อมูลเชิงลึกเฉพาะตัวสินค้า เช่น ชื่อ, ราคา, รายละเอียดผ้า, สีทั้งหมด, สถานะไซส์พร้อมสต็อกคงคลัง, ลิงก์รูปภาพตารางไซส์สินค้า (Size Chart) และกลุ่มรูปภาพของสินค้าเพิ่มเติมจากแกลเลอรี (โดยทำการกรองรูปภาพซ้ำและรูปตารางไซส์ออก) เพื่อเตรียมส่งเข้าฐานข้อมูล โดยจะนำผลลัพธ์มาจัดระเบียบใส่ตาราง Pandas DataFrame ให้อ่านและตรวจสอบได้อย่างง่ายดาย

### ขั้นตอนที่ 6.1: สร้างฟังก์ชันดึงรายละเอียดสินค้าเชิงลึก (Scrape Product Detail Helper)
สร้างฟังก์ชัน `scrape_product_detail` สำหรับดึง ชื่อสินค้า ราคาสิ่งแรก คำบรรยายลักษณะผ้า (USP) รายการสี รายการไซส์ สถานะสต็อก ภาพตารางไซส์เสื้อ (Size Chart) และภาพแกลเลอรีสินค้าเพิ่มเติม (Gallery Images)

In [50]:
import sys
import asyncio
import threading
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import re

def scrape_product_detail(url: str):
    """
    ดึงรายละเอียดเจาะลึกเฉพาะหน้าสินค้า เช่น สี, ไซส์, ตรวจสถานะสต็อกสินค้า, ตารางไซส์ และรูปแกลเลอรี
    """
    detail_data = {}
    exception = None

    def worker():
        nonlocal detail_data, exception
        try:
            if sys.platform == 'win32':
                asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
                
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            
            async def run():
                async with async_playwright() as p:
                    browser = await p.chromium.launch(headless=True)
                    context = await browser.new_context(
                        user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
                    )
                    page = await context.new_page()
                    await page.set_viewport_size({"width": 1280, "height": 800})
                    
                    print(f"Navigating to product detail: {url}")
                    await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                    await page.wait_for_timeout(4000) # รอข้อมูลเรนเดอร์เพิ่ม
                    
                    html_content = await page.content()
                    soup = BeautifulSoup(html_content, "html.parser")
                    
                    # 1. ดึงชื่อสินค้า
                    title_tag = soup.find("h1")
                    if not title_tag:
                        title_tag = soup.find(lambda tag: tag.name in ["p", "div"] and "Ultrasoft" in tag.text and len(tag.text) < 150)
                    product_name = title_tag.get_text(strip=True) if title_tag else "Yuedpao Premium Product"
                    
                    # 2. ดึงราคา
                    price_text = ""
                    price_p = soup.find("p", class_=lambda x: x and "text-ci-primary" in x)
                    if price_p:
                        price_text = price_p.get_text(strip=True)
                    else:
                        p_fallback = soup.find(lambda tag: tag.name in ["p", "div", "span"] and "฿" in tag.text and len(tag.text) < 30)
                        if p_fallback:
                            price_text = p_fallback.get_text(strip=True)
                    
                    price = re.sub(r"[^\d]", "", price_text)
                    price = int(price) if price else 0
                    
                    # 3. ดึงคำอธิบายข้อเด่นผ้า (USP Description)
                    desc_text = ""
                    p_desc = soup.find(lambda tag: tag.name == "p" and len(tag.text) > 40 and "ยับยาก" in tag.text)
                    if not p_desc:
                        p_desc = soup.find(lambda tag: tag.name == "p" and len(tag.text) > 30)
                    desc_text = p_desc.get_text(strip=True) if p_desc else "ผ้านุ่มใส่สบาย ยืดแต่ไม่ย้วย ยับยาก ไม่ต้องรีด"
                    if len(desc_text) > 60:
                        desc_text = desc_text[:57] + "..."
                        
                    # 4. ดึงสีและไซส์คงเหลือ
                    buttons = soup.find_all("button")
                    colors = []
                    sizes = {}
                    size_pattern = re.compile(r"^(XS|S|M|L|XL|[2-9]XL)$", re.IGNORECASE)
                    
                    for btn in buttons:
                        btn_text = btn.get_text(strip=True)
                        classes = btn.get("class", [])
                        is_disabled = "Mui-disabled" in classes
                        
                        if any(c in classes for c in ["mr-3", "mt-2", "min-w-[100px]"]):
                            if size_pattern.match(btn_text):
                                sizes[btn_text] = not is_disabled
                            else:
                                if btn_text and btn_text != "เข้าสู่ระบบ":
                                    colors.append(btn_text)
                                
                    # 5. รูปสินค้าหลัก
                    image_url = ""
                    img_div = soup.find("div", style=lambda x: x and "background-image" in x)
                    if img_div and "style" in img_div.attrs:
                        style_str = img_div["style"]
                        match = re.search(r'url\((?:&quot;|"|\')?(.*?)(?:&quot;|"|\')?\)', style_str)
                        if match:
                            image_url = match.group(1)
                            
                    # 6. รูปตารางไซส์ (Size Chart Image)
                    size_chart_url = ""
                    size_chart_img = soup.find("img", class_=lambda x: x and "mpe-no-image-placeholder" in x)
                    if size_chart_img:
                        size_chart_url = size_chart_img.get("src", "")
                            
                    # 7. รูปแกลเลอรีสินค้าเพิ่มเติม (Gallery Images - กรองลิงก์รูปหลัก ตารางไซส์ และคำที่เกี่ยวกับตารางไซส์ออก)
                    gallery_images = []
                    divs = soup.find_all("div", style=lambda x: x and "background-image" in x)
                    for div in divs:
                        style_str = div.get("style", "")
                        match = re.search(r'url\((?:&quot;|"|\')?(.*?)(?:&quot;|"|\')?\)', style_str)
                        if match:
                            g_url = match.group(1)
                            # เก็บเฉพาะ URL ที่มีคีย์เวิร์ด galleries และไม่ซ้ำกับภาพหลัก/ตารางไซส์ และไม่มีคีย์เวิร์ดของตารางขนาด
                            if g_url and "galleries" in g_url:
                                if (g_url != image_url and 
                                    g_url != size_chart_url and 
                                    "size" not in g_url.lower() and 
                                    "chart" not in g_url.lower() and 
                                    g_url not in gallery_images):
                                    gallery_images.append(g_url)
                            
                    result_dict = {
                        "name": product_name,
                        "price": price,
                        "description": desc_text,
                        "colors": colors,
                        "sizes": sizes,
                        "image_url": image_url,
                        "size_chart_url": size_chart_url,
                        "gallery_images": gallery_images,
                        "product_url": url,
                        "is_available": any(sizes.values()) if sizes else False
                    }
                    await browser.close()
                    return result_dict
                    
            detail_data = loop.run_until_complete(run())
        except Exception as e:
            exception = e
        finally:
            loop.close()

    thread = threading.Thread(target=worker)
    thread.start()
    thread.join()
    
    if exception:
        raise exception
    return detail_data

### ขั้นตอนที่ 6.2: สั่งรันดึงข้อมูลจริงและจัดแสดงผลลัพธ์เป็นตาราง (Run Detail Scraper & Display DataFrame)
ทดสอบรันเพื่อดึงข้อมูลจากหน้ารายละเอียดของสินค้าตัวอย่าง และนำค่าที่คุณสมบัติดึงได้มาจัดแสดงบนตาราง Pandas DataFrame เพื่อให้ตรวจสอบค่าได้ง่าย

In [51]:
import pandas as pd

# URL สินค้าเชิงลึกตัวอย่าง
sample_detail_url = "https://www.yuedpao.com/physical/Ultrasoft-Unisex-Round-Neck-%E0%B8%AA%E0%B8%B5-Pink-Cotton-azmu24o6p9b9911c939d"

print("Starting Scraper Product Detail...")
product_detail_data = scrape_product_detail(sample_detail_url)

print("\n--- ดึงข้อมูลสินค้าสำเร็จ! จัดทำตารางสรุปผล ---")

# แปลงดิกชันนารีไซส์ให้เป็นสตริง
size_statuses = [f"{sz}: {'มี' if avail else 'หมด'}" for sz, avail in product_detail_data.get("sizes", {}).items()]
sizes_str = ", ".join(size_statuses) if size_statuses else "N/A"

# แปลงกลุ่มสีให้เป็นสตริง
colors_str = ", ".join(product_detail_data.get("colors", []))

# แปลงรายการภาพแกลเลอรีให้เป็นสตริงแสดงหลายบรรทัด
gallery_str = "\n".join([f"- {img}" for img in product_detail_data.get("gallery_images", [])]) if product_detail_data.get("gallery_images") else "ไม่มีรูปเพิ่มเติม"

table_data = {
    "คุณสมบัติสินค้า (Attribute)": [
        "ชื่อสินค้า (Name)",
        "ราคา (Price)",
        "คำอธิบาย / USP (Description)",
        "สีทั้งหมดที่เจอ (Colors)",
        "ไซส์คงเหลือ (Sizes Stock Status)",
        "ลิงก์รูปหลัก (Main Image URL)",
        "ลิงก์ตารางไซส์ (Size Chart Image URL)",
        "รูปภาพแกลเลอรีเพิ่มเติม (Gallery Images)",
        "ลิงก์หน้าสินค้าจริง (Product URL)",
        "พร้อมจำหน่ายหรือไม่ (Is Available)"
    ],
    "ค่าที่ดึงได้ (Value)": [
        product_detail_data.get("name"),
        f"฿{product_detail_data.get('price')}",
        product_detail_data.get("description"),
        colors_str or "N/A",
        sizes_str,
        product_detail_data.get("image_url"),
        product_detail_data.get("size_chart_url"),
        gallery_str,
        product_detail_data.get("product_url"),
        "ใช่" if product_detail_data.get("is_available") else "ไม่ (หมดสต็อกทุกไซส์)"
    ]
}

# ปิดการย่อขนาดตัวอักษรของลิงก์ใน Pandas เพื่อให้ลิงก์แสดงแบบเต็มๆ ไม่ถูกย่อด้วย ...
pd.set_option('display.max_colwidth', None)

df_detail = pd.DataFrame(table_data)
display(df_detail)

Starting Scraper Product Detail...


Navigating to product detail: https://www.yuedpao.com/physical/Ultrasoft-Unisex-Round-Neck-%E0%B8%AA%E0%B8%B5-Pink-Cotton-azmu24o6p9b9911c939d

--- ดึงข้อมูลสินค้าสำเร็จ! จัดทำตารางสรุปผล ---


,คุณสมบัติสินค้า (Attribute),ค่าที่ดึงได้ (Value)
0,ชื่อสินค้า (Name),Ultrasoft Unisex Round Neck สี Pink Cotton
1,ราคา (Price),฿100
2,คำอธิบาย / USP (Description),ผ้านุ่มใส่สบาย ยืดแต่ไม่ย้วย ยับยาก รีดง่าย เป็นรุ่นสร้าง...
3,สีทั้งหมดที่เจอ (Colors),Pink Cotton
4,ไซส์คงเหลือ (Sizes Stock Status),"S: หมด, M: หมด, L: หมด, XL: หมด, 2XL: หมด, 3XL: หมด, 4XL: หมด, 5XL: หมด, 6XL: หมด"
5,ลิงก์รูปหลัก (Main Image URL),https://mp-static.yuedpao.com/physical/cover/azmu24o6p9b9911c939d/image/hw36ivkx
6,ลิงก์ตารางไซส์ (Size Chart Image URL),https://mp-static.yuedpao.com/physical/azmu24o6p9b9911c939d/image/5dr6phj5
7,รูปภาพแกลเลอรีเพิ่มเติม (Gallery Images),- https://mp-static.yuedpao.com/physical/galleries/azmu24o6p9b9911c939d/1fv2ulcdh0uk99xijl55/image/x2nxcoks\n- https://mp-static.yuedpao.com/physical/galleries/azmu24o6p9b9911c939d/lhreuah3bnl9480eeshh/image/ynwd2esm\n- https://mp-static.yuedpao.com/physical/galleries/azmu24o6p9b9911c939d/vb211ff7d3jbzsxmhrv4/image/ue8z4hlw\n- https://mp-static.yuedpao.com/physical/mainAttribute/azmu24o6p9b9911c939d/galleries/8p736uhnq7p6gfbfi5aj/image/vdn9k9gv
8,ลิงก์หน้าสินค้าจริง (Product URL),https://www.yuedpao.com/physical/Ultrasoft-Unisex-Round-Neck-%E0%B8%AA%E0%B8%B5-Pink-Cotton-azmu24o6p9b9911c939d
9,พร้อมจำหน่ายหรือไม่ (Is Available),ไม่ (หมดสต็อกทุกไซส์)
